## Amazon SageMaker Tutorial for end-to-end ML 

Amazon SageMaker is AWS' end-to-end ML platform, allowing you to build, train, tune, deploy, and monitor ML models without managing servers. 
- It is like Jupyter + GPU instances + AutoML + model hosting + MLOps tools all bundled into one AWS Services. 
- Covers all key stages of the ML workflow: 
    - [Preprocessing]: Launch Jupyter notebooks in the cloud (with S3 and IAM built-in)
    - [Training]: Train models at scale on CPU/GPU clusters, spot instances 
    - [Hyperparam tuning]: Built in Bayesian optimisation for model tuning 
    - [Deployment]: Deploy models as fully managed APIs (no Flask, EC2 hassle)
    - [Monitoring]: Track drift, latency, prediction quality in real time 
    - [Pipelines]: Build end-to-end ML pipelines with retraining, evaluation, deployment 
- Core SageMaker components 
    - Notebook Instance: cloud hosted Jupyter Notebooks 
    - Training Job: run training on managed infra
    - Model artifact: stores the trained model (model.tar.gz)
    - Endpoint: Deployed REST API for inference 
    - Pipeline: CI/CD for retraining and redeploying 
    - Ground Truth: Labelling service for building datasets 
    - Feature Store: Versioned, centralised store of ML Features
- Under the hood, SageMaker runs on Docker containers, but AWS abstracts it 

Assume you already have data in S3. Now, there are two main development workflows for Sagemaker, but in both cases, S3 = central storage hub, and SageMaker uses it to read/write inputs and outputs
- [Most common pattern in production teams] Develop locally but define estimator etc. to upload onto SageMaker container to launch training remotely.
    -Code lives in my local machine but training runs in SageMaker-manged container (EC2 under the hood)
        - Container is a lightweight, self-contained box that packages your code, dependencies and environment
            - E.g. pre-cooked meal that includes everything it needs to work, just heat and run 
        - EC2 is Elastic Compute Cloud, AWS service for launching VMs 
            - EC2 is similar to renting a computer in the cloud for a certain time period 
    - Requires SageMaker Python SDK + AWS credentials on machine 
        - AWS credentials are set (using aws configure)
        - Data (e.g. train.csv) is in s3://my-bucket/data/train.csv
        - Have boto3 and sagemaker Python SDK (software development kit) installed, if not run pip install boto3 sagemaker pandas scikit-learn
    - Uploads train.py, source_dir, data to S3
    - Use this when want full local control
 - [SageMaker Studio/Notebook] Develop directly on SageMaker by logging into AWS Console, launching a SageMaker notebook instance, and write + run everything in cloud. 
    - Code lives in a Jupyter notebook in AWS
    - Training runs in SageMaker container or directly inside the notebook kernel with EC2 instances with access to S3
    - Does not require local AWS setup, just browser access 
    - Great for ad-hoc experiments, GPU access, or if want to avoid local setup headaches

Key objects 
- session = sagemaker.Session() is critical as this creates the session object and stores it in a variable that needs to be passed into the estimator when I actually launch the training job. This is required as the Estimator needs to 
    - Upload script (session.upload_data() internally)
    - Get default bucket (session.default_bucket())
    - Track job status (session.logs_for_job())
    - Deploy model (session.create_model_package())
    - Note: The beauty that that I don't need to call any methods on session itself, just pass it in when defining the estimator
- Estimator: high-level abstration in SageMaker that defines everything needed to train a model in the cloud. 
    - It bundles together: 
        - Code configuration 
            - Main training script to execute inside the container (entry_point) 
            - Folder containing helper scripts and files used in main training script (source_dir)
        - Compute Resources 
            - instance_type: Type of EC2 instance (e.g. ml.m5.large, ml.p3.2xlarge) 
            - instance_count: No. of machines (1 usually unless doing distributed training )
        - What framework to use 
            - The framework-specific Estimator class you choose determines the environment
                - E.g. PyTorch(), TensorFlow(), SKLearn(), XGBoost()
                - This is not an argument passed into Estimator per se, but rather determines which class of Estimator you use when I instantiate the object 
                - e.g. 
                    - from sagemaker.pytorch import PyTorch 
                    - estimator = PyTorch(...)
            - framework_version: version of the chosen ML framework to use (e.g. 1.13.1 for PyTorch)
        - Training Configuration
            - hypermarameters: dict of hyperparam values (e.g. {'lr': 0.01, 'epochs': 10})
        - Input/Output 
            - fit() input: path to the training data in S3, passed when you launch the job via estimator.fit({'train': 's3://my-bucket/datasets/iris}). 
                - This dictionary maps input channels (label that identifies what the data is used for e.g. train, validation, test, inference) to the S3 URI (full S3 path to dataset I want to use for that channel). Channel is essentially what role the dataset plays in model development and training 
                    - Hence, think of the fit() dictionary as a declaration of input roles
                    - If pass multiple channels, then my script can then do train_path = os.environ['SM_CHANNEL_TRAIN'], val_path = os.environ['SM_CHANNEL_VALIDATION']
                        - fit({'train': 's3://bucket/train_data',
                                'validation': 's3://bucket/val_data'
                                })
                    - Can use any channel name in .fit(), as long as I match it exactly in train.py 
                - When I do this, SageMaker 
                    - Downloads S3 folder into container and maps it to this directory in the container: /input/data/train/
                    - Creates environment variable: os.environ['SM_CHANNEL_TRAIN'] == '/opt/ml/input/data/train'
                        - Which I can then access in train.py 
            - output_path in S3; where the trained model artifacts will be saved (e.g. s3://my-bucket/model-output)
        - Other required inputs 
            - role: the IAM role that gives SageMaker permission to access S3 and other AWS services 
            - sagemaker_session: a sagemaker.Session() object that manages interaction with SageMaker, S3, and other AWS resources 


Other notes 
- Can monitor training jobs and endpoints in the AWS Console -> SageMaker -> Training jobs / Endpoints 
- Can use SageMaker Pipelines for full MLOps 
- Workflow for online AWS notebook dev: 
    1. Launch a SageMaker notebook (Notebook Instance)
    2. Load the data from S3 
    3. Train a model with SageMaker by starting a training job 
    4. Savve model output to S3
    5. Deploy the trained model as an API endpoint 
    6. Do real-time inference by sending inference requests via boto3

#### Overview of key code (but in segments)
- For deployment code use folder structure as per below

In [ ]:
#Step 1: Setup and Session 
import boto3 #AWS official Python SDK to interact with any AWS service programmatically
import sagemaker #AWS SageMaker Python SDK that wraps around boto3, providing higher-level abstractions (e.g. Session, Estimator, Predictor)
from sagemaker import get_execution_role #this is a helper function from sagemaker SDK that automatically fetches the IAM execution role associated with my SageMaker Notebook or instance, thereby providing necessary permissions for SageMaker jobs 
from sagemaker.sklearn.estimator import SKLearn #this is SageMaker's pre-built scikit-learn estimator class that lets me train scikit-learn models on SM's managed infrastructure. This handles uploading of training script to S3, launching training job in a container, saving trained model, and optionally deploying it as an API

session = sagemaker.Session() #session is an object representing my SM's runtime context to manage interactions with S3 and job tracking. session object is later passed into my estimator when I actually launch the training job 
role = get_execution_role() #connect my training job to an IAM role to obtain necessary permissions

In [ ]:
#Step 2: Load Data from S3
bucket = 'my-bucket'
prefix = 'data/train.csv'
s3_uri = f's3://{bucket}/{prefix}' #knowing the bucket and key (prefix) completely specifies the s3_url for the desired file. i.e. s3://<bucket>//<path/to/file_or_folder>

print("Training Data Location:", s3_uri)

#If want to preview data
import pandas as pd 
df = pd.read_csv(s3_uri) #pandas can read directly from S3 as long as I give it a valid S3 URI, and have been properly authenticated
df.head()

In [ ]:
#Step 3: Write training script (save this as train.py in working directory)

##train.py 
import pandas as pd 
from sklearn.ensemble import RandomForestClassifier
import joblib
import os 

if __name__ == "__main__": 
    data = pd.read_csv(os.path.join('/opt/ml/input/data/train', 'train.csv')) #loads train.csv from the SageMaker's training input directory. Note that os.path.join() safely builds file paths - no hardcoding slashes or assuming OS
    X = data.drop('label', axis=1) #splits dataset into features (X) and target (y), latter assumed to be parked under the label column
    y = data['label']

    model = RandomForestClassifier()
    model.fit(X,y)

    joblib.dump(model, '/opt/ml/model/model.joblib')  # Saves model in binary format to the SageMaker output directory so that can upload to S3 afterward

In [ ]:
#Step 4: Define and Launch Training Job. 

estimator = SKLearn( #creates SM Training job definition using SKLearn Estimator class from SM SDK
    entry_point='train.py', #this is the Python file you want to run inside training container
    role=role, #IAM role which gives SM permission to access S3, write logs etc
    instance_type = 'ml.m5.large', #tells SM what kind of machine to use to train the model. Options include 'ml.m5.large': general purpose, 'ml.c5.xlarge': compute-optimized, 'ml.p3.2xlarge': for GPUs (deep learning)
    framework_version = '1.2-1', #locks down the scikit-learn version that SM installs inside the container. Must match what I need in the train.py script
    sagemaker_session = session, #links job to current SM session, which knows (1) region, (2) where to store logs, (3) where to upload trained model in S3
    base_job_name = 'rf_train' #name prefix SM use for training job, model artifact in S3, CloudWatch logs
)

estimator.fit({"train": s3_uri}) #This code (1) spins up a container in AWS infra using ml.m5.large, installs the scikit-learn version specified, and mounts /opt/ml/input/data/train to your S3 training data, (2) runs train.py script that loads data, trains model, and saves model, (3) Saves the model back to S3
#output is a trained model stored in S3 that can be deployed later using estimator.deploy()

In [ ]:
#Step 5: Deploy the model 
## This gives live HTTPS endpoint (i.e. the actual server) for inference. Live HTTPS endpoint is a secure web API hosted by AWS where I send in JSON/numpy arrays, it feeds into trained model and returns prediciton result
#  Takes trained model sitting in S3 and spinning up an actual server in cloud that (1) loads model, (2) listens for HTTPs requests, (3) returns predictions in real-time
# Hence, what you get is (1) SageMaker Model (created from trained model.tar.gz in S3), (2) SageMaker Endpoint (fully managed REST API that can send requests to), (3) Predictor Object (Python wrapper to talk to endpoint directly)
# Note that when the endpoint is running, you are billed by the second for the EC2 instance and should delete when done
# Also note that this HTTPS endpoint is not public like a normal website, but is secured by AWS IAM permissions so only authenticated users or services can invoke it

predictor = estimator.deploy(
    initial_instance_count=1, 
    instance_type="ml.m5.large"
)

In [ ]:
#Step 6: Run inference 
import numpy as np 
test_data = np.array([[5.1, 3.5, 1.4, 0.2]])  # shape must match training features
result = predictor.predict(test_data)
print("Prediction:", result)

In [ ]:
#Step 7: Clean up to avoid overcharging 
predictor.delete_endpoint()

#### Further notes on REST API 
- REST API (Representational State Transfer API) is a web interface that lets different systems talk to each other over HTTP, usually by sending and receiving a JSON 
    - Analogy: A waiter (API) takes order (request), passes to kitchen (model or backend) and sends the food (response) - all over the internet
    - REST is a design style for APIs over HTTP 
    - API: interface for two systems to communicate 
- REST APIs are used for 
    - Connecting frontends (React, Streamlit) to backends (Flask, FastAPI, SageMaker)
    - Letting apps integrate with models 
    - Automating pipelines: tools hit APIs to send or get results 
    - Replacing UI: you interact using code not clicks
- Common REST API Methods 
    - GET: read something (e.g. get user profile info)
        - E.g. when visit webpage, it loads via a GET request because I am asking the server to send me some content (HTML, CSS, JS, images) so that browser cna display it 
        - Get request asks the server for something, and server responds with that content
    - POST: submit data (e.g. send text to be classified)
        - Used when want to send data to server, usually to create, submit, or trigger something
        - E.g. submitting form (posting form data), making prediction from model (posting input data), uploading photo (posting image file)
            - Note: if doing real-time inference e.g. POST to /predict, usually do not need a follow-up GET as model prediction is returned immediately as part of POST response
    - PUT: Update something (change a password)
    - DELETE: remove something (delete image or file)
- FASTAPI: modern Python web framework for building REST APIs - perfect for serving ML Models in real time 
    - Same as Flask, but faster and more powerful out of the box, with built-in validation and auto-documentation via Swagger
        - Swagger UI is a visual, interactive API documentation that FastAPI auto-generates for you from your code. It lets you 
            - See available endpoints (e.g. /predict)
            - Understand input and output formats 
            - Try out the API right in the browser
        - Don't need to install anything; when FASTAPI server is running, just visit http://localhost:8000/docs
        - Schema in FASTAPI + Swagger is a formal definition of what your API expects and returns (i.e. contract between frontend and backend)
            - More specifically, schema is a contract that defines what keys your JSON (or dict) must have, what types (e.g. int, list[int], bool) the values must be, and what shape, format, or constraints those values must follow

#### Further notes on joblib 
joblib is a Python library for 
- Saving and loading trained models 
- Serialising big data structures (e.g. NumPy arrays)
- Optional: parallel processing jobs 

It is more efficient than pickle (extension .plk vs .joblib), being the fastest and most memory-efficient choice for scikit-learn models, pipelines, and NumPy-heavy objects

Two key functions 
- joblib.dump(obj, file) -> Saves Python object to disk (e.g. trained model)
    - file: full fule path where SageMaker expects to find the trained model so it can package and upload to S3
- joblib.load(file) -> restores the object e.g. during before inference to serve via API

In [ ]:
## Example: FastAPI endpoint 
from fastapi import FastAPI #imports FastAPI framework to define route e.g. /predict, and handle HTTP requests
from pydantic import BaseModel #Pydantic used for data validation, BaseModel lets you define expected input formats (e.g. schemas). This is built into FastAPI for automatic request parsing and type checking
import joblib #use joblib to load saved models which was earlier trained 

app = FastAPI() #creates a FastAPI application instance -> think of this as the server
model = joblib.load("model.joblib") #load the trained model from disk into memory, can be any scikit-learn compatible model

class Input(BaseModel): #expected format of your input data when calling the /predict API. Without this, have to parse JSON manually and write error-handling boilerplate
    values: list #this schema says that I expect a POST request body that contains a JSON object with a key values, and it must be a list. E.g. {"values": [5.1, 3.5, 1.4, 0.2]} is acceptable

@app.post("/predict") #defines a post route called /predict.  #when someone sends a post request to predict, this funciton will run
def predict(input: Input): #input argument automatically populated by FASTApi using the Input class defined with BaseModel (Pydantic). It parses the JSON body, validates it and converts it into a Python object
    result = model.predict([input.values]) #model expects 2D array so wrap input.values in [] (additional square brackets)
    return {"prediction": int(result[0])} #result[0] is the model's predicted label, int ensures its JSON-serialisable as some models return numpy.int64. Result is usually a np.ndarray


#### Folder Structure 

Typically we organise the above code into a project folder with .py scripts

sagemaker-ml-project/
- data/
    - train.csv
- scripts/         (python scrips used in training or interence)
    - train.py     (training script)
- src/
    - __init__.py  (makes src/ a Python package, so can import src.utils from anywhere)
    - utils.py     (helper functions including loading data, logging, splitting datasets etc)
- tests/ 
    - test_utils.py (test file containing unit test)
- notebooks/
    - sagemaker_train.ipynb (jupyter notebooks for dev/experimentation)
- models/ (optional: local saved models or outputs)
- config/
    - params.json (config files that store hyperparameters etc)
- requirements.txt (python dependencies)
- README.md (project overview)
- run.py (script to orchestrate training/deploy locally)


#### Important note on distinction between developing project locally and running on Sagemaker
- I develop and organise my project locally (on my laptop or in VS Code with the above folder structure),
- However, when I launch a SageMaker training job, SageMaker will spin up a container with its own internal file structure below. 
    - This is regardless of what internal folder structure I have, the container will always have the same structure
- Hence, my job runs inside SageMaker's world, not directly in my local folder layout

What happens under the hood 
1. My script(s) get zipped and uploaded to S3 
    - entry_point = the script SageMaker will run (train.py)
    - source_dir = the additional files to include (e.g. src/, config/, requirements.txt)
2. SageMaker spins up a Docker container on a virtual machine in the cloud 
3. It downloads my zipped code into /opt/ml/code/
4. It runs python /opt/ml/code/train.py, and gives 
    - /opt/ml/input/data/train -> training data from S3
    - /opt/ml/model/ -> job should write the final model here

Final internal structure in the container 
/opt/ml/
- code/  (uploaded project code)
    - train.py
    - src/
    - config/
- input/    
    - data/
        - train/ (training data (CSV, images etc))
- model/ (code writes trained model here)

Hence, we must adjust the path to the model and training data dependinig on whether we are running / testing locally or on SageMaker
- I.e. the code is adaptive so that it works both locally and on SageMaker 
- This is recommended by using Environment Variables (see below)
    - SM_CHANNEL_TRAIN = /opt/ml/input/data/train
    - SM_MODEL_DIR = /opt/ml/model


In [ ]:
#scripts/train.py 
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
import joblib
import os

if __name__ == "__main__":
    input_path = os.environ.get("SM_CHANNEL_TRAIN", "data") #fallback to local path ("data/"), unless running on SageMaker which sets environment variables. This also assumes that my CWD (current working directory) is already the root of my project (sagemaker-ml-project/), i.e. where the data/ folder lives
    output_path = os.environ.get("SM_MODEL_DIR", "models") #os.environ is a special dict-like object in Python that represents env variables available to process. Hence can use .get() to safely retrieve value corresponding to specified key, or returns fallback

    df = pd.read_csv(os.path.join(input_path, 'train.csv'))
    X = df.drop('label', axis=1)
    y = df['label']

    model = RandomForestClassifier()
    model.fit(X, y)

    joblib.dump(model, os.path.join(output_path, 'model.joblib'))


In [ ]:
#run.py (for local launching)
from sagemaker.sklearn.estimator import SKLearn
import sagemaker
from sagemaker import get_execution_role

session = sagemaker.Session()
role = get_execution_role()

estimator = SKLearn(
    entry_point='scripts/train.py',
    role=role,
    instance_type='ml.m5.large',
    framework_version='1.2-1',
    sagemaker_session=session,
    base_job_name='rf-train'
)

s3_input = 's3://your-bucket/data/train.csv'
estimator.fit({'train': s3_input})

predictor = estimator.deploy(initial_instance_count=1, instance_type='ml.m5.large')


In [ ]:
#src/utils.py (Sample Functions)
import os
import pandas as pd
from sklearn.model_selection import train_test_split

def load_csv(file_path):
    """Loads CSV from local or S3 path."""
    return pd.read_csv(file_path)

def save_model(model, path):
    """Saves model to specified path using joblib."""
    import joblib
    os.makedirs(os.path.dirname(path), exist_ok=True)
    joblib.dump(model, path)

def split_data(df, target_column, test_size=0.2, random_state=42):
    """Splits DataFrame into train/test sets."""
    X = df.drop(columns=[target_column])
    y = df[target_column]
    return train_test_split(X, y, test_size=test_size, random_state=random_state)

def log(msg):
    """Basic logger."""
    print(f"[INFO] {msg}")


In [ ]:
#src/__init__.py 
# This file makes src a Python package
# Optional: import useful things to top-level
from .utils import load_csv, save_model, split_data, log


In [ ]:
#test/test_utils.py (have to install pytest and add it to requirements.txt)

import pandas as pd
import os
import tempfile
import joblib
from src.utils import load_csv, save_model, split_data, log

def test_load_csv():
    df = pd.DataFrame({'a': [1, 2], 'b': [3, 4]})
    with tempfile.NamedTemporaryFile(suffix='.csv', delete=False) as tmp:
        df.to_csv(tmp.name, index=False)
        loaded = load_csv(tmp.name)
        assert loaded.equals(df)
        os.remove(tmp.name)

def test_save_model():
    model = {"foo": "bar"}
    with tempfile.TemporaryDirectory() as tmpdir:
        path = os.path.join(tmpdir, 'model.joblib')
        save_model(model, path)
        assert os.path.exists(path)
        loaded = joblib.load(path)
        assert loaded == model

def test_split_data():
    df = pd.DataFrame({
        'feature1': [1, 2, 3, 4],
        'feature2': [5, 6, 7, 8],
        'label':    [0, 1, 0, 1]
    })
    X_train, X_test, y_train, y_test = split_data(df, target_column='label', test_size=0.5)
    assert len(X_train) == len(y_train) == 2
    assert len(X_test) == len(y_test) == 2

def test_log(capfd):
    log("Hello world")
    out, _ = capfd.readouterr()
    assert "[INFO] Hello world" in out
